# Python — Parte 6: Classes e Objetos

Anotações para estudo e revisão: criação de classes, instâncias, atributos, métodos, herança e composição. Os exemplos foram reorganizados e reescritos a partir dos temas do material autoral, com exercícios resolvidos e testes simples.

### Materiais complementares

- [Orientação a objetos — material de referência](./reference_materials/07_orientacao_a_objetos.pdf)
- [Orientação a objetos — cheat sheet](./cheat_sheets/cheat_sheet_orientacao_a_objetos.pdf)


## 1. Criando classes e instâncias

Uma classe define a estrutura e o comportamento de objetos. `__init__` inicializa o estado de cada instância; `self` representa a própria instância. Os métodos operam sobre esse estado.

In [1]:
class Cachorro:
    def __init__(self, nome, idade):
        self.nome = nome
        self.idade = idade

    def apresentar(self):
        return f'{self.nome} tem {self.idade} ano(s).'

    def aniversario(self):
        self.idade += 1

luna = Cachorro('Luna', 2)
max = Cachorro('Max', 5)
assert luna.apresentar() == 'Luna tem 2 ano(s).'
luna.aniversario()
assert luna.idade == 3 and max.idade == 5
print(luna.apresentar(), max.apresentar())

Luna tem 3 ano(s). Max tem 5 ano(s).


## 2. Alterando atributos com métodos

Métodos podem proteger regras de negócio: ao atualizar a quilometragem de um carro, podemos rejeitar valores negativos ou menores que a leitura atual. Atributos que começam com `_` seguem uma convenção de uso interno, mas isso não equivale a proteção absoluta.

In [2]:
class Carro:
    def __init__(self, marca, modelo, ano):
        self.marca = marca
        self.modelo = modelo
        self.ano = ano
        self._quilometragem = 0

    @property
    def quilometragem(self):
        return self._quilometragem

    def atualizar_quilometragem(self, valor):
        if valor < self._quilometragem:
            raise ValueError('Não é permitido reduzir a quilometragem')
        self._quilometragem = valor

    def rodar(self, distancia):
        if distancia < 0:
            raise ValueError('A distância não pode ser negativa')
        self._quilometragem += distancia

    def descricao(self):
        return f'{self.marca} {self.modelo} ({self.ano})'

carro = Carro('Toyota', 'Corolla', 2020)
carro.rodar(120)
carro.atualizar_quilometragem(200)
assert carro.quilometragem == 200
print(carro.descricao(), carro.quilometragem)

Toyota Corolla (2020) 200


## 3. Herança e sobrescrita de métodos

Uma subclasse pode reutilizar métodos da classe base e alterar o comportamento de métodos específicos (*override*). `super()` chama a implementação da classe pai. Use herança quando a relação “é um tipo de” for apropriada.

In [3]:
class CarroEletrico(Carro):
    def __init__(self, marca, modelo, ano, autonomia_km):
        super().__init__(marca, modelo, ano)
        self.autonomia_km = autonomia_km

    def descricao(self):
        return f'{super().descricao()} — elétrico, autonomia {self.autonomia_km} km'

eletrico = CarroEletrico('Nissan', 'Leaf', 2021, 250)
eletrico.rodar(30)
assert eletrico.quilometragem == 30
print(eletrico.descricao())

Nissan Leaf (2021) — elétrico, autonomia 250 km


## 4. Composição — instâncias como atributos

Na composição, um objeto possui outro objeto como atributo: um carro elétrico **tem uma bateria**, ao contrário da herança, em que um carro elétrico **é um carro**.

In [4]:
class Bateria:
    def __init__(self, capacidade_kwh):
        self.capacidade_kwh = capacidade_kwh

    def estimar_autonomia(self, consumo_kwh_100km):
        if consumo_kwh_100km <= 0:
            raise ValueError('Consumo deve ser positivo')
        return self.capacidade_kwh / consumo_kwh_100km * 100

class VeiculoEletrico(Carro):
    def __init__(self, marca, modelo, ano, capacidade_kwh):
        super().__init__(marca, modelo, ano)
        self.bateria = Bateria(capacidade_kwh)

veiculo = VeiculoEletrico('Exemplo', 'E1', 2025, 60)
assert veiculo.bateria.estimar_autonomia(15) == 400
print('Autonomia estimada:', veiculo.bateria.estimar_autonomia(15), 'km')

Autonomia estimada: 400.0 km


## 5. Referência rápida

| Conceito | Papel |
|---|---|
| `class` | Define classe |
| `__init__` | Inicialização da instância |
| `self` | Referência à instância |
| `@property` | Exposição controlada de atributo |
| `super()` | Acesso ao comportamento da classe base |
| Herança | Relação “é um tipo de” |
| Composição | Relação “tem um” |

## 6. Exercícios resolvidos

**Exercício 1 — Conta bancária.** Crie uma classe com saldo privado por convenção, depósito e saque que não permitam valores não positivos ou saldo insuficiente.

In [5]:
class Conta:
    def __init__(self, titular, saldo_inicial=0):
        if saldo_inicial < 0:
            raise ValueError('Saldo inicial inválido')
        self.titular = titular
        self._saldo = saldo_inicial

    @property
    def saldo(self):
        return self._saldo

    def depositar(self, valor):
        if valor <= 0:
            raise ValueError('Depósito deve ser positivo')
        self._saldo += valor

    def sacar(self, valor):
        if valor <= 0 or valor > self._saldo:
            raise ValueError('Valor de saque inválido')
        self._saldo -= valor

conta = Conta('Ana', 100)
conta.depositar(50)
conta.sacar(80)
assert conta.saldo == 70
print(conta.titular, conta.saldo)

Ana 70


**Exercício 2 — Biblioteca.** Modele um livro e o registro de empréstimo, mantendo o estado de disponibilidade. Explique a diferença entre atributos e métodos.

In [6]:
class Livro:
    def __init__(self, titulo, autor):
        self.titulo, self.autor = titulo, autor
        self.disponivel = True

    def emprestar(self):
        if not self.disponivel:
            return False
        self.disponivel = False
        return True

    def devolver(self):
        self.disponivel = True

livro = Livro('Introdução a Python', 'Autor fictício')
assert livro.emprestar() is True
assert livro.emprestar() is False
livro.devolver()
assert livro.disponivel is True
print(livro.titulo, 'disponível:', livro.disponivel)

Introdução a Python disponível: True


## 7. Para revisar

Classes agrupam dados e operações; a escolha entre herança e composição depende da relação entre objetos. Evite criar classes para tudo: funções e estruturas simples são suficientes em muitos problemas.

## Aprofundamento 1 — `dataclass` e invariantes
`@dataclass` reduz código repetitivo, mas a validação de regras ainda é responsabilidade da classe. `__post_init__` funciona como ponto de verificação após a inicialização.

In [7]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Produto:
    nome: str
    preco: float

    def __post_init__(self):
        if not self.nome.strip() or self.preco < 0:
            raise ValueError("Nome obrigatório e preço não negativo")

    def com_desconto(self, percentual: float) -> float:
        if not 0 <= percentual <= 1:
            raise ValueError("Percentual inválido")
        return round(self.preco * (1 - percentual), 2)

produto = Produto("Livro", 50)
assert produto.com_desconto(.2) == 40
print(produto)

Produto(nome='Livro', preco=50)


## Aprofundamento 2 — Propriedades e encapsulamento
Uma propriedade pode garantir que alterações preservem as regras de negócio. Em Python, `_saldo` indica atributo de uso interno por convenção, sem ser uma barreira de segurança.

In [8]:
class Carteira:
    def __init__(self, saldo_inicial=0):
        if saldo_inicial < 0:
            raise ValueError("Saldo inicial inválido")
        self._saldo = float(saldo_inicial)

    @property
    def saldo(self) -> float:
        return self._saldo

    def depositar(self, valor: float) -> None:
        if valor <= 0:
            raise ValueError("Depósito deve ser positivo")
        self._saldo += valor

    def sacar(self, valor: float) -> None:
        if valor <= 0 or valor > self._saldo:
            raise ValueError("Saque inválido")
        self._saldo -= valor

carteira = Carteira(100)
carteira.depositar(25)
carteira.sacar(30)
assert carteira.saldo == 95
print("Saldo:", carteira.saldo)

Saldo: 95.0


## Aprofundamento 3 — Composição, protocolos e polimorfismo
Use composição quando um objeto **possui** outro. Para intercambiar comportamentos, classes distintas podem implementar a mesma interface pública sem herdar da mesma classe.

In [9]:
class FreteFixo:
    def calcular(self, peso_kg: float) -> float:
        return 12.0

class FretePorPeso:
    def calcular(self, peso_kg: float) -> float:
        return 5.0 + 2.0 * peso_kg

def preco_total(valor: float, peso: float, estrategia) -> float:
    return valor + estrategia.calcular(peso)

assert preco_total(100, 3, FreteFixo()) == 112
assert preco_total(100, 3, FretePorPeso()) == 111
print("Estratégias de frete verificadas")

Estratégias de frete verificadas


## Exercício integrador com solução — catálogo e carrinho
Modele itens com validação, use composição e confirme quantidade e total. Evite misturar cálculo, impressão e entrada interativa no mesmo método.

In [10]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ItemCarrinho:
    produto: Produto
    quantidade: int

    def __post_init__(self):
        if self.quantidade <= 0:
            raise ValueError("Quantidade deve ser positiva")

class Carrinho:
    def __init__(self):
        self.itens: list[ItemCarrinho] = []

    def adicionar(self, item: ItemCarrinho) -> None:
        self.itens.append(item)

    def total(self) -> float:
        return sum(item.produto.preco * item.quantidade for item in self.itens)

carrinho = Carrinho()
carrinho.adicionar(ItemCarrinho(Produto("Livro", 50), 2))
carrinho.adicionar(ItemCarrinho(Produto("Caneta", 3), 3))
assert carrinho.total() == 109
print("Total do carrinho:", carrinho.total())

Total do carrinho: 109


### Para aprofundar
- Imutabilidade reduz efeitos colaterais, mas nem todo modelo deve ser congelado.
- Composição normalmente é mais flexível que uma hierarquia profunda de herança.
- Anotações de tipo e testes se complementam: nenhuma delas substitui validação em tempo de execução.
- Evite depender de detalhes internos de outras classes; prefira interfaces pequenas e claras.